In [3]:
import os
import sys

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", "..", ".."))
sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))
print(f"PROJECT_ROOT: {PROJECT_ROOT}")

from pyspark.sql import SparkSession

from src.helu.utils.config import BRONZE_DIR, GOLD_DIR, SILVER_DIR

spark = (
    SparkSession.builder.master("local[1]")
    .appName("QueryTables")
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.1")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "2")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark: {spark.version}")

PROJECT_ROOT: /Users/seleneandrade/Documents
Spark: 3.5.6


## Bronze tables

In [6]:
# Query Bronze tables

b_exchange_df = spark.read.format("delta").load(f"{BRONZE_DIR}/exchange_rates")
b_exchange_df.createOrReplaceTempView("b_exchange_rates")
print(f"Bronze Exchange Rates columns: {b_exchange_df.columns}")
print("-------------------------------")

b_apfel_df = spark.read.format("delta").load(f"{BRONZE_DIR}/apfel")
b_apfel_df.createOrReplaceTempView("b_apfel")
print(f"Bronze Apfel columns: {b_apfel_df.columns}")
print("-------------------------------")

b_fenster_df = spark.read.format("delta").load(f"{BRONZE_DIR}/fenster")
b_fenster_df.createOrReplaceTempView("b_fenster")
print(f"Bronze Fenster columns: {b_fenster_df.columns}")
print("-------------------------------")

Bronze Exchange Rates columns: ['date', 'currency', 'rate_to_eur', '_ingestion_timestamp_utc', '_source', '_inserted_date_utc']
-------------------------------
Bronze Apfel columns: ['event_id', 'event_timestamp', 'event_type', 'customer_uuid', 'customer_email', 'customer_created_at', 'country_code', 'region', 'city', 'postal_code', 'subscription_type', 'renewal_period', 'currency', 'amount', 'tax_amount', 'discount_code', 'affiliate_id', 'device_type', 'app_version', 'session_id', 'internal_ref', 'processing_status', '_ingestion_timestamp_utc', '_source', '_inserted_date_utc']
-------------------------------
Bronze Fenster columns: ['id', 'ts', 'type', 'cid', 'mail', 'signup_ts', 'ctry', 'state', 'zip', 'plan', 'ccy', 'price', 'tax', 'vat_id', 'campaign_src', 'utm_medium', 'utm_campaign', 'browser', 'os', 'screen_res', 'lang', 'tz', 'legacy_flag', 'migrated_from', 'batch_id', 'row_hash', '_ingestion_timestamp_utc', '_source', '_inserted_date_utc']
-------------------------------


In [8]:
query = """
    Select * from b_apfel
"""

spark.sql(query).show()

+---------------+--------------------+--------------------+-------------+----------------+--------------------+------------+--------+---------+-----------+-----------------+--------------+--------+------+----------+-------------+------------+-----------+-----------+-----------+------------+-----------------+------------------------+-------+------------------+
|       event_id|     event_timestamp|          event_type|customer_uuid|  customer_email| customer_created_at|country_code|  region|     city|postal_code|subscription_type|renewal_period|currency|amount|tax_amount|discount_code|affiliate_id|device_type|app_version| session_id|internal_ref|processing_status|_ingestion_timestamp_utc|_source|_inserted_date_utc|
+---------------+--------------------+--------------------+-------------+----------------+--------------------+------------+--------+---------+-----------+-----------------+--------------+--------+------+----------+-------------+------------+-----------+-----------+----------

### Silver tables

In [14]:
# Query Silver tables

s_exchange_df = spark.read.format("delta").load(f"{SILVER_DIR}/exchange_rates")
s_exchange_df.createOrReplaceTempView("s_exchange_rates")
print(f"Silver Exchange Rates columns: {s_exchange_df.columns}")
print("-------------------------------")

s_apfel_df = spark.read.format("delta").load(f"{SILVER_DIR}/apfel")
s_apfel_df.createOrReplaceTempView("s_apfel")
print(f"Silver Apfel columns: {s_apfel_df.columns}")
print("-------------------------------")

s_fenster_df = spark.read.format("delta").load(f"{SILVER_DIR}/fenster")
s_fenster_df.createOrReplaceTempView("s_fenster")
print(f"Silver Fenster columns: {s_fenster_df.columns}")
print("-------------------------------")

Silver Exchange Rates columns: ['date', 'currency', 'rate_to_eur', '_ingestion_timestamp_utc', '_source', '_inserted_date_utc', '_silver_ingestion_timestamp_utc']
-------------------------------
Silver Apfel columns: ['event_id', 'event_timestamp', 'event_type', 'customer_id', 'email', 'signup_timestamp', 'country_code', 'region', 'city', 'postal_code', 'subscription_plan', 'renewal_period', 'currency', 'price_amount', 'tax_amount', 'discount_code', 'affiliate_id', 'device_type', 'app_version', 'session_id', 'internal_ref', 'processing_status', '_ingestion_timestamp_utc', '_source', '_inserted_date_utc', '_silver_ingestion_timestamp_utc', 'subscription_status', 'currency_norm']
-------------------------------
Silver Fenster columns: ['event_id', 'event_timestamp', 'subscription_status', 'customer_id', 'email', 'signup_timestamp', 'country', 'state', 'zip_code', 'original_subscription_plan', 'currency', 'price_amount', 'tax_amount', 'vat_id', 'campaign_source', 'utm_medium', 'utm_campai

In [15]:
query = """
    Select * from s_exchange_rates
"""

spark.sql(query).show()

+----------+--------+-----------+------------------------+--------------+------------------+-------------------------------+
|      date|currency|rate_to_eur|_ingestion_timestamp_utc|       _source|_inserted_date_utc|_silver_ingestion_timestamp_utc|
+----------+--------+-----------+------------------------+--------------+------------------+-------------------------------+
|2025-02-01|     USD|       0.92|    2026-03-14 16:44:...|exchange_rates|        2026-03-14|           2026-03-14 16:51:...|
|2025-03-01|     USD|       0.91|    2026-03-14 16:44:...|exchange_rates|        2026-03-14|           2026-03-14 16:51:...|
|2025-04-01|     USD|        0.9|    2026-03-14 16:44:...|exchange_rates|        2026-03-14|           2026-03-14 16:51:...|
|2025-05-01|     USD|       0.89|    2026-03-14 16:44:...|exchange_rates|        2026-03-14|           2026-03-14 16:51:...|
|2025-06-01|     USD|       0.88|    2026-03-14 16:44:...|exchange_rates|        2026-03-14|           2026-03-14 16:51:...|


### Gold report


In [11]:
mrr_df = spark.read.format("delta").load(f"{GOLD_DIR}/mrr_financial_report")
mrr_df.createOrReplaceTempView("mrr_report")
print(f"MRR Report columns: {mrr_df.columns}")
print("-------------------------------")

MRR Report columns: ['platform', 'subscription_type', 'country', 'report_month', 'original_currency', 'acquisitions', 'renewals', 'cancellations', 'mrr_eur', '_gold_ingestion_timestamp_utc', '_inserted_date_utc']
-------------------------------


In [12]:
query = """
    Select * from mrr_report
"""
spark.sql(query).show()

+--------+-----------------+-------+------------+-----------------+------------+--------+-------------+------------------+-----------------------------+------------------+
|platform|subscription_type|country|report_month|original_currency|acquisitions|renewals|cancellations|           mrr_eur|_gold_ingestion_timestamp_utc|_inserted_date_utc|
+--------+-----------------+-------+------------+-----------------+------------+--------+-------------+------------------+-----------------------------+------------------+
|   apfel|          premium|     DE|     2025-07|              EUR|         171|     190|           19| 4397.999999999977|         2026-03-14 15:00:...|        2026-03-14|
|   apfel|         standard|     DE|     2025-07|              EUR|          66|      55|           44| 914.1000000000006|         2026-03-14 15:00:...|        2026-03-14|
|   apfel|          premium|     GB|     2025-07|              EUR|          35|       7|            7| 552.7900000000002|         2026-03-1